# Daily airport closure hours

Two line charts: **total hours across all six airports**, followed by **one colored line per airport**. All dates use Moscow time. Hover over a point for its hours. Dashed segments and faded points mark provisional periods; gaps mean unavailable data.

[Open the cancellations report](airport_analysis.ipynb). Refresh with **Kernel → Restart & Run All**.

The current Moscow day is excluded because its totals are incomplete.

In [1]:
from pathlib import Path
from datetime import datetime, timedelta
from html import escape
import json
import math
import sqlite3
from IPython.display import HTML, display
from collect_restrictions import MSK
from restriction_report import report_data, daily_restrictions

# Change this to show a longer history.
DAYS = 14
ROOT = Path.cwd()
DB = ROOT / 'flights.sqlite3'
if not DB.is_file():
    raise FileNotFoundError('Start Jupyter in the airports project directory.')
REPORT_AIRPORTS = {'SVO', 'DME', 'VKO', 'LED', 'KZN', 'AER'}
config = [a for a in json.loads((ROOT / 'airports.json').read_text()) if a['iata'] in REPORT_AIRPORTS]
codes = [a['iata'] for a in config]
colors = ['#2563eb', '#ea580c', '#16a34a', '#9333ea', '#dc2626', '#0891b2']
with sqlite3.connect(f'file:{DB}?mode=ro', uri=True) as con:
    con.row_factory = sqlite3.Row
    try:
        closure_rows, closure_cutoff, _ = report_data(con, config, scope='tracked', kind='restricted')
        closure_error = None
        closure_records = con.execute('SELECT * FROM airport_restrictions').fetchall()
        closure_by_airport = {
            code: daily_restrictions(closure_records, closure_cutoff, airports={code}, kind='restricted')
            for code in codes
        }
    except (sqlite3.OperationalError, ValueError) as exc:
        closure_rows, closure_cutoff = [], None
        closure_error = str(exc)
closures = {r['date']: r for r in closure_rows}
available_days = sorted(closures)
# Exclude the current Moscow day: its totals are still incomplete.
today_msk = datetime.now(MSK).date()
available_days = [day for day in available_days if datetime.fromisoformat(day).date() < today_msk]
if available_days:
    end_day = datetime.fromisoformat(available_days[-1]).date()
    # Show the latest closure history on the same dates in both charts.
    history_start = datetime.fromisoformat(min(closures) if closures else available_days[0]).date()
    start_day = max(history_start, end_day - timedelta(days=DAYS - 1))
    dates = [(start_day + timedelta(days=i)).isoformat() for i in range((end_day - start_day).days + 1)]
else:
    dates = []

def line_chart(series, title, unit, provisional):
    if not dates:
        return '<p>No closure history available yet.</p>'
    width, height = 1000, 450
    left, right, top, bottom = 60, 975, 65, 350
    values = [v for _, data, _ in series for day in dates if (v := data.get(day)) is not None]
    peak = max(values + [1])
    rough = peak / 4
    magnitude = 10 ** math.floor(math.log10(rough))
    step = next(s * magnitude for s in (1, 2, 5, 10) if s * magnitude >= rough)
    ymax = math.ceil(peak / step) * step
    svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" role="img" aria-label="{escape(title)}" style="display:block;width:100%;height:auto;font:13px system-ui;background:white">',
           f'<text x="60" y="25" font-size="20" font-weight="600">{escape(title)}</text>',
           f'<text x="60" y="47" fill="#64748b">{escape(unit)}</text>']
    for tick in range(int(ymax / step) + 1):
        value = tick * step
        y = bottom - (bottom - top) * value / ymax
        svg.append(f'<path d="M{left},{y} H{right}" stroke="#e2e8f0"/><text x="{left-10}" y="{y+4}" text-anchor="end">{value:g}</text>')
    spacing = (right-left) / max(len(dates)-1, 1)
    xs = [left+i*spacing if len(dates)>1 else (left+right)/2 for i in range(len(dates))]
    for i, day in enumerate(dates):
        if i % max(math.ceil(len(dates)/10), 1) == 0 or i == len(dates)-1:
            label = datetime.fromisoformat(day).strftime('%b %d').replace(' 0', ' ')
            svg.append(f'<text x="{xs[i]}" y="{bottom+25}" text-anchor="middle">{label}</text>')
    # A gap breaks the line; missing observations never become zeros.
    for j, (code, data, color) in enumerate(series):
        previous = None
        for i, day in enumerate(dates):
            value = data.get(day)
            if value is None:
                previous = None
                continue
            x, y = xs[i], bottom - (bottom-top)*value/ymax
            partial = provisional(day, code)
            if previous is not None:
                px, py, was_partial = previous
                dash = ' stroke-dasharray="6 4"' if partial or was_partial else ''
                svg.append(f'<path d="M{px},{py} L{x},{y}" fill="none" stroke="{color}" stroke-width="2.5"{dash}/>')
            detail = f'{day} · {code}: {value:.2f} {unit.lower()}' + (' (provisional)' if partial else '')
            svg.append(f'<circle cx="{x}" cy="{y}" r="4" fill="{color}" fill-opacity="{.5 if partial else 1}"><title>{escape(detail)}</title></circle>')
            previous = x, y, partial
        lx = left + j*145
        svg.append(f'<path d="M{lx},410 h24" stroke="{color}" stroke-width="3"/><circle cx="{lx+12}" cy="410" r="4" fill="{color}"/><text x="{lx+32}" y="414">{escape(code)}</text>')
    svg.append('</svg>')
    return ''.join(svg)
names = ' · '.join(a['iata'] + ' — ' + a['name'] for a in config)
period = f'{dates[0]} to {dates[-1]}' if dates else 'No data'
display(HTML(f'<p><b>{escape(period)}</b> · Latest {DAYS} days available<br>{escape(names)}</p>'))
display(HTML(f'<p style="color:#64748b">Closures last imported: {escape(closure_cutoff.isoformat() if closure_cutoff else "unavailable")}</p>'))

## Total closure hours

Sum across SVO, DME, VKO, LED, KZN and AER. Two airports closed for 3 hours each contribute **6 airport-hours**. Only closed (`restricted`) periods count; partial operating restrictions are excluded. Overlaps are merged per airport, and periods crossing midnight are split by date. Ongoing closures stop at the latest successful import.

In [2]:
if closure_error:
    display(HTML('<p>Closure history is unavailable. Run collect_restrictions.py, then rerun this notebook.</p>'))
else:
    closure_hours = {day: row['restriction_hours'] for day, row in closures.items()
                     if row['restriction_status'] != 'outside source history'}
    airport_hours = {
        code: {day: closure_by_airport[code].get(day, {}).get('restriction_hours', 0.0)
               for day in closure_hours}
        for code in codes
    }
    for day, total in closure_hours.items():
        assert math.isclose(sum(airport_hours[code][day] for code in codes), total,
                            rel_tol=1e-9, abs_tol=1e-9), f'Closure sum mismatch on {day}'
    display(HTML(line_chart([('Total', closure_hours, '#172033')],
                           'Total closure hours — six airports', 'Airport-hours',
                           provisional=lambda day, _: closures[day]['restriction_status'] != 'completed intervals')))

## Closure hours by airport

One line per airport, using the same colors as the cancellations report. Zero means no recorded closure within the source history.

In [3]:
if not closure_error:
    series = [(code, airport_hours[code], colors[i % len(colors)]) for i, code in enumerate(codes)]
    display(HTML(line_chart(series, 'Daily closure hours by airport', 'Hours closed',
                           provisional=lambda day, code: (
                               closures[day]['restriction_status'] != 'completed intervals'
                               or closure_by_airport[code].get(day, {}).get('restriction_status') == 'provisional'))))